# E2 encoder runs on Colab (D12)

Six steps:

1. Put `bundle_<name>.zip` in the Google Drive folder `MyDrive/azd_colab/` (create the folder if needed).
2. Open this notebook in Colab (File > Upload notebook, or open it from Drive).
3. Runtime > Change runtime type > Hardware accelerator: GPU (L4 or A100) > Save.
4. Key icon in the left bar (Secrets) > Add new secret: name `HF_TOKEN`, value = your Hugging Face token; switch on "Notebook access". (Needed for the gated `allmalab/bert-base-aze`.)
5. Runtime > Run all. Allow Google Drive access when asked.
6. If Colab disconnects, reconnect (same GPU type) and Run all again: finished fits are kept and the run continues. When the last cell prints "results zipped", the results are in `MyDrive/azd_colab/results_<name>.zip`.

Do not change any setting below except `BUNDLE_NAME` ("random", then "benchmark"). The protocol settings (learning-rate grid, epochs, batch 16, length 512, top-k, seeds) are the defaults of `sys_encoder.py` and are not passed or changed here.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/azd_colab"   # folder with bundle_<name>.zip; results go here too
BUNDLE_NAME = "random"    # "random" first, then "benchmark" (see HOWTO_colab.md); "" = the only bundle_*.zip in DRIVE_DIR
PRIORITIES = [1, 2, 3, 4, 5, 6]   # D12 order 1-4, then 5 = C0 (run guide), 6 = human-only labels. Do not reorder.
MICRO_BATCH = 16          # GPU memory chunk of the batch of 16 (batch stays 16: gradients are accumulated)
ROOT = "/content/azd"     # where the bundle is unzipped (local disk, rebuilt every session)
C0_BEFORE_EDITS = False   # False: priority 5 waits while priority 4 has no edit set (D12 order)
ERRORS = []                # failures of single priorities are collected here; the run goes on and the zip cell reports them


## (a) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import glob, os
os.makedirs(DRIVE_DIR, exist_ok=True)
zips = sorted(glob.glob(f"{DRIVE_DIR}/bundle_*.zip"))
if BUNDLE_NAME:
    ZIP = f"{DRIVE_DIR}/bundle_{BUNDLE_NAME}.zip"
elif len(zips) == 1:
    ZIP = zips[0]
else:
    raise SystemExit(f"found {len(zips)} bundles in {DRIVE_DIR}: put exactly one there, or set BUNDLE_NAME")
NAME = os.path.basename(ZIP)[len("bundle_"):-len(".zip")]
OUT = f"{DRIVE_DIR}/results_{NAME}"
print("bundle:", ZIP, "\nresults folder:", OUT)

## (b) Unzip the bundle to /content and check every file's sha256

In [ ]:
import shutil, sys, zipfile
shutil.rmtree(ROOT, ignore_errors=True)
zipfile.ZipFile(ZIP).extractall(ROOT)
sys.path.insert(0, ROOT)
import importlib, run_e2 as R
R = importlib.reload(R)     # use the runner just unzipped, even in a runtime that imported an older one
MAN = R.verify_bundle(ROOT)

## (c) Install the exact library versions recorded in the bundle manifest

In [ ]:
import subprocess, sys
assert sys.version_info >= (3, 12), f"Python {sys.version}: the pinned numpy/scipy need Python >= 3.12"
reqs = R.requirements(ROOT)
print("installing:", " ".join(reqs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
# Colab preinstalls torchaudio/torchvision/peft built for another CUDA; transformers then fails on import
# (seen 10 Nov: "PyTorch and TorchAudio were compiled with different CUDA versions"). Not used by this project.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio", "torchvision", "peft"], check=False)
chk = subprocess.run([sys.executable, "-c", "from transformers import get_linear_schedule_with_warmup, AutoModelForSequenceClassification"],
                     capture_output=True, text=True)
if chk.returncode != 0:
    raise SystemExit("transformers import check failed after install:\n" + chk.stderr[-2000:])
print("transformers import check OK")
mm = R.check_versions(ROOT)
if mm:
    raise SystemExit("version mismatch after install: " + "; ".join(mm))
print("library versions match the manifest (pip warnings about other Colab packages can be ignored)")

## (d) Hugging Face token (Colab secret HF_TOKEN)


In [ ]:
import os
from google.colab import userdata

# The AzBERT checkpoint is gated on Hugging Face: add your token as the Colab secret HF_TOKEN
# (key icon in the left bar, switch on "Notebook access").
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.setdefault("HF_HOME", "/content/hf_cache")

from huggingface_hub import hf_hub_download
hf_hub_download("allmalab/bert-base-aze", "config.json",
                revision="5fd824965138a261552fc5ca333e91f4fa45cc67", token=os.environ["HF_TOKEN"])
print("Hugging Face access to AzBERT works")

## (e) Check the GPU, bind the results folder, load the data once

In [ ]:
gpu = R.cuda_info()
print("GPU:", gpu)
if gpu == "none":
    raise SystemExit("no GPU: Runtime > Change runtime type > GPU (L4 or A100), then Run all again")
R.prepare_out(ROOT, OUT)
R.preflight(ROOT)

### Priority 1: XLM-R and AzBERT, C2, natural
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 1 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 1, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 1: {e}")
        print("!!! priority 1 did not finish:", e, "-> continuing")


### Priority 2: mDeBERTa-v3, C2, natural
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 2 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 2, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 2: {e}")
        print("!!! priority 2 did not finish:", e, "-> continuing")


### Priority 3: all three, C1, natural
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 3 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 3, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 3: {e}")
        print("!!! priority 3 did not finish:", e, "-> continuing")


### Priority 4: all three with the edit set (C2, then C1)
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 4 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 4, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 4: {e}")
        print("!!! priority 4 did not finish:", e, "-> continuing")


### Priority 5: all three, C0, natural
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 5 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 5, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 5: {e}")
        print("!!! priority 5 did not finish:", e, "-> continuing")


### Priority 6: all three, C2, trained on human labels only (prereg 4a secondary variant)
Every finished fit is written to the Drive results folder at once; re-running this cell resumes.

In [ ]:
if 6 in PRIORITIES:
    try:
        R.run_priority(ROOT, OUT, 6, MICRO_BATCH, require_cuda=True, c0_before_edits=C0_BEFORE_EDITS)
    except (SystemExit, Exception) as e:          # keep going: the next priority still runs, finished fits stay on Drive
        ERRORS.append(f"priority 6: {e}")
        print("!!! priority 6 did not finish:", e, "-> continuing")


### H7 selection (only if the bundle carries the Mac logreg/gbm outputs; otherwise H7 is run locally)

In [ ]:
H7_ERROR = None
try:
    R.run_h7(ROOT, OUT)
except (SystemExit, Exception) as e:
    H7_ERROR = e      # reported after the results are zipped
    ERRORS.append(f"H7: {e}")

## (f) Zip the results on Drive

In [ ]:
import json, os
try:
    R.zip_results(OUT, f"{DRIVE_DIR}/results_{NAME}.zip")
    if os.path.exists(f"{OUT}/progress.json"):
        print(json.dumps(json.load(open(f"{OUT}/progress.json")), indent=1))
    else:
        print("no progress.json yet (the run stopped before the first priority)")
finally:
    drive.flush_and_unmount()
print("Drive flushed. Send back:", f"{DRIVE_DIR}/results_{NAME}.zip")
errs = globals().get("ERRORS", [])
if errs:
    print("\nSOME STEPS DID NOT FINISH (their logs are in the zip; Run all again resumes them):")
    for e in errs:
        print(" -", e)
    raise SystemExit(f"{len(errs)} step(s) did not finish; send the zip anyway")
print("ALL PRIORITIES FINISHED")